In [ ]:
import pandas as pd
import requests
from dotenv import load_dotenv
import os
import psycopg2
import json

#### Définition de l'accès

In [ ]:
load_dotenv()

USERNAME = os.getenv("INPI_USERNAME")
PASSWORD = os.getenv("INPI_PASSWORD")

if not USERNAME or not PASSWORD:
    raise RuntimeError("INPI_USERNAME ou INPI_PASSWORD manquant dans le .env")

LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"

payload = {
    "username": USERNAME,
    "password": PASSWORD,
}

response = requests.post(LOGIN_URL, json=payload, timeout=10)
response.raise_for_status()

token = response.json()["token"]

HEADERS = {
    "Authorization": f"Bearer {token}"
}

#### Vérification de l'accès

In [ ]:
print("Token OK :", token[:20], "...")

---

#### Test de réponse de l'API pour vérifier la présence de bilans

In [ ]:
# On définit le SIREN à tester
test_siren = "389091893"

# On construit l'adresse (URL)
url_test = f"https://registre-national-entreprises.inpi.fr/api/companies/{test_siren}/attachments"

# On lance la requête
response_test = requests.get(url_test, headers=HEADERS, timeout=10)

# On vérifie si ça a marché
if response_test.status_code == 200:
    data = response_test.json()
    print("✅ Succès ! 'data' a été mémorisé.")

    print("Contenu disponible :", data.keys())
else:
    print(f"Erreur {response_test.status_code} : {response_test.text}")

In [ ]:
# Extraction du CA (Code BJ) du bilan le plus récent
try:
    dernier_bilan = data['bilansSaisis'][0]['bilanSaisi']['bilan']['detail']['pages'][0]['liasses']
    
    # On cherche le code 'BJ' dans la liste
    ca_info = next((item for item in dernier_bilan if item["code"] == "BJ"), None)
    
    if ca_info:
        ca_valeur = int(ca_info['m1'])
        print(f"💰 Chiffre d'Affaires (BJ) trouvé : {ca_valeur} €")
    else:
        print("❌ Code BJ non trouvé dans ce bilan.")
        
except (KeyError, IndexError) as e:
    print(f"❌ Erreur lors de l'extraction : {e}")

##### Vérifications des bilans disponibles sur la structure choisie

In [ ]:
# --- Comptage des bilans pour ce SIREN ---

# 1. Bilans totaux (inclut les PDF)
nb_bilans_pdf = len(data.get('bilans', []))

# 2. Bilans exploitables (données structurées)
nb_bilans_saisis = len(data.get('bilansSaisis', []))

print(f"📊 Pour le SIREN {data.get('siren', test_siren)} :")
print(f"   - Nombre total de dépôts (PDF) : {nb_bilans_pdf}")
print(f"   - Nombre de bilans numérisés (exploitables) : {nb_bilans_saisis}")

# 3. Détail des années disponibles en mode "Saisi"
if nb_bilans_saisis > 0:
    annees = [b['dateCloture'][:4] for b in data['bilansSaisis']]
    print(f"   - Années disponibles : {', '.join(annees)}")

#### Acquisition des 3 bilans au format json

In [ ]:
# --- Extraction structurée pour OCB (Format Dictionnaire Optimisé) ---

bilans_propres = []

if "bilansSaisis" in data:
    for b in data["bilansSaisis"]:
        # Extraction des métadonnées de base
        info_bilan = {
            "siren": b.get("siren"),
            "date_cloture": b.get("dateCloture"),
            "confidentialite": b.get("confidentiality"),
            "liasses": {} # <-- On passe en dictionnaire ici
        }
        
        try:
            pages = b['bilanSaisi']['bilan']['detail']['pages']
            for page in pages:
                for liasse in page['liasses']:
                    code = liasse.get("code")
                    # On convertit en int, par sécurité on met 0 si m1 est absent ou None
                    valeur = int(liasse.get("m1") or 0)
                    
                    if code:
                        # On range directement : "BJ": 138904
                        info_bilan["liasses"][code] = valeur
            
            bilans_propres.append(info_bilan)
        except KeyError:
            continue

# --- Visualisation du résultat ---
print(f"✅ {len(bilans_propres)} bilans extraits pour OCB.")

# Aperçu du format dictionnaire
import json
print("Structure du premier bilan (extrait) :")
print(json.dumps(bilans_propres[0], indent=2, ensure_ascii=False)[:500] + "...")

# Sauvegarde physique sur le disque
with open("test_ocb.json", "w", encoding="utf-8") as f:
    json.dump(bilans_propres, f, indent=2, ensure_ascii=False)

print("\n💾 Le fichier 'test_ocb.json' (format dictionnaire) a été créé.")

---

#### Connection BDD

In [ ]:
# Charger .env
load_dotenv()

# Récupérer l'URL
DATABASE_URL = os.getenv("NEON_DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("La variable NEON_DATABASE_URL n'est pas définie dans .env")

try:
    # On ajoute sslmode=require pour Neon si ce n'est pas déjà dans ton URL
    conn = psycopg2.connect(DATABASE_URL)
    cur = conn.cursor()
    
    # Un petit test pour confirmer la version de la BDD
    cur.execute("SELECT version();")
    db_version = cur.fetchone()
    print(f"✅ Connexion à Néon réussie !")
    print(f"🖥️ Version : {db_version[0][:30]}...")

except Exception as e:
    print(f"❌ Erreur de connexion : {e}")

In [ ]:
# 1. On s'assure que la table existe
create_table_query = """
CREATE TABLE IF NOT EXISTS bilans_entreprises (
    id SERIAL PRIMARY KEY,
    siren VARCHAR(9) NOT NULL,
    date_cloture DATE NOT NULL,
    confidentialite TEXT,
    donnees_liasses JSONB,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    UNIQUE(siren, date_cloture)
);
"""

try:
    cur.execute(create_table_query)
    # OPTIONNEL : On vide la table pour repartir sur le nouveau format propre
    cur.execute("TRUNCATE TABLE bilans_entreprises;") 
    conn.commit() 
    print("🏗️ Table 'bilans_entreprises' prête et vidée pour le nouveau format.")
except Exception as e:
    conn.rollback()
    print(f"❌ Erreur lors de la préparation de la table : {e}")

# 2. Injection des données OCB (format dictionnaire)
from psycopg2.extras import Json

insert_query = """
INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
VALUES (%s, %s, %s, %s)
ON CONFLICT (siren, date_cloture) DO NOTHING;
"""

try:
    for bilan in bilans_propres:
        cur.execute(insert_query, (
            bilan['siren'],
            bilan['date_cloture'],
            bilan['confidentialite'],
            Json(bilan['liasses']) # Enverra le dictionnaire propre {"BJ": 138904, ...}
        ))
    conn.commit()
    print(f"🚀 Succès ! Les {len(bilans_propres)} bilans d'OCB sont dans Neon au format exploitable.")
except Exception as e:
    conn.rollback()
    print(f"❌ Erreur d'injection : {e}")